# A Persian Legal RAG Pipeline

Retrieval-augmented generation over seven Iranian legal codes — labour law, cheque
law, VAT, landlord-and-tenant, social security, deeds registration, and
anti-smuggling — built as a LangGraph state machine over a LanceDB vector store,
and evaluated with RAGAS.

The pipeline is deliberately more than retrieve-then-generate. Six nodes:
query rewriting, intent classification, metadata extraction, filtered retrieval,
reranking, and grounded generation, with a retry edge when reranking finds nothing
relevant.

**RAGAS: faithfulness 0.734, answer relevancy 0.514** over 21 questions spanning
all seven codes.

The gap between those two numbers is the finding. Faithfulness at 0.73 means
answers are mostly grounded in the retrieved text — the anti-hallucination
instructions work. Relevancy at 0.51 means they often ground themselves in the
*wrong* text. The diagnosis is in section 5: the metadata pre-filter never
matches, so every query silently falls back to unfiltered vector search.

In [ ]:
from pathlib import Path

# Data is resolved relative to the repository root, so the notebook runs the
# same whether Jupyter was started here or one level up.
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA = ROOT / 'data'

## 1. Loading the OCR model

`DeepSeek-OCR` served through vLLM, for the scanned PDFs in the corpus.

In [ ]:
import os
from huggingface_hub import snapshot_download
from vllm import LLM, SamplingParams

model_id = "deepseek-ai/DeepSeek-OCR"
local_model_path = "deepseek-ai/DeepSeek-OCR" 

print("Checking for existing files on HDD")
try:
    snapshot_download(
        repo_id=model_id,
        local_dir=local_model_path,
        local_dir_use_symlinks=False, 
        resume_download=True          
    )
    print(f"Model files are ready in {local_model_path}")
except Exception as e:
    print(f"Note: Could not verify all files online, will try to load anyway. Error: {e}")


try:
    print("Loading DeepSeek-OCR from HDD")
    llm = LLM(
        model=local_model_path, 
        trust_remote_code=True,
        max_model_len=8192,
        gpu_memory_utilization=0.7,
        enforce_eager=True 
    )
    print("Model loaded successfully from External Drive.")
except Exception as e:
    print(f"Error loading model: {e}")

### The embedding model

`intfloat/multilingual-e5-small` — multilingual, so it embeds Persian legal text
without a Persian-specific model.

In [ ]:
import os
from huggingface_hub import snapshot_download


model_id = "intfloat/multilingual-e5-small" 


local_e5_path = "intfloat/multilingual-e5-base"

print(f"Checking/Downloading {model_id} to {local_e5_path}...")


snapshot_download(
    repo_id=model_id,
    local_dir=local_e5_path,
    local_dir_use_symlinks=False,  
    resume_download=True           
)

print(" Model files are ready.")

### A quick OCR check

In [ ]:
import torch
from PIL import Image
from vllm import LLM, SamplingParams
from vllm.model_executor.models.deepseek_ocr import NGramPerReqLogitsProcessor
import io

## 2. Ingesting seven legal codes

The dispatch here is the useful part. Each PDF is probed for a selectable text
layer, and only the ones without go through OCR:

- **5 of 7 PDFs are digital text** → extracted with `pypdf` in milliseconds.
- **2 of 7 are scanned images** → routed to DeepSeek-OCR at 300 DPI.

The alternative — OCR everything — would have been slower and *less* accurate,
since OCR on a clean digital PDF introduces errors the text layer does not have.

In [ ]:
import os
import glob
import torch
import gc
from vllm import LLM, SamplingParams
from pdf2image import convert_from_path
from pypdf import PdfReader


model_path = "deepseek-ai/DeepSeek-OCR"
pdf_folder = str(DATA / 'laws') 


gc.collect()
torch.cuda.empty_cache()


print("Loading OCR Model")
llm = LLM(
    model=model_path,
    trust_remote_code=True,
    max_model_len=4096,         
    gpu_memory_utilization=0.8,  
    swap_space=4,                
    enforce_eager=True,
    tensor_parallel_size=1
)


def is_selectable_pdf(pdf_path):
    try:
        reader = PdfReader(pdf_path)
        text_content = ""
       
        for i, page in enumerate(reader.pages[:3]):
            text = page.extract_text()
            if text:
                text_content += text
        
       
        if len(text_content.strip()) > 50:
            return True
        return False
    except:
        return False

def extract_text_standard(pdf_path):
    reader = PdfReader(pdf_path)
    full_text = ""
    for page in reader.pages:
        extracted = page.extract_text()
        if extracted:
            full_text += extracted + "\n"
    return full_text

def extract_text_ocr(pdf_path):
    print("   ↳ Converting PDF to High-Res Images (300 DPI)")
    images = convert_from_path(pdf_path, dpi=300) 
    
    full_text = ""
    prompt = "<image>\n<|grounding|>Convert to markdown."
    
    inputs = []
    for img in images:
        inputs.append({
            "prompt": prompt,
            "multi_modal_data": {"image": img}
        })
        
    
    sampling_params = SamplingParams(
        temperature=0.0,         
        max_tokens=4096,        
        repetition_penalty=1.8,  
        skip_special_tokens=True
    )
    
    outputs = llm.generate(inputs, sampling_params=sampling_params)
    for output in outputs:
        full_text += output.outputs[0].text + "\n\n"
        
    return full_text


pdf_files = glob.glob(f"{pdf_folder}/**/*.pdf", recursive=True)
print(f" Found {len(pdf_files)} PDFs.")

for pdf_path in pdf_files:
    txt_path = pdf_path.replace(".pdf", ".txt")
    filename = os.path.basename(pdf_path)
    
    print(f"------------------------------------------------")
    print(f" Processing: {filename}")
    
    try:
        if is_selectable_pdf(pdf_path):
            print("    Type: Digital Text (Selectable)")
            print("    Using Fast Extraction (pypdf)...")
            final_text = extract_text_standard(pdf_path)
        else:
            print("    Type: Scanned Image")
            print("    Using DeepSeek-OCR (Slow & Powerful)...")
            final_text = extract_text_ocr(pdf_path)
            
   
        with open(txt_path, "w", encoding="utf-8") as f:
            f.write(final_text)
            
        print(f" Saved to: {os.path.basename(txt_path)}")
        
    except Exception as e:
        print(f" Failed to process {filename}: {e}")

print(" All tasks finished!")

## 3. The vector store

LanceDB with the E5 embedding function registered against the schema, so LanceDB
embeds on insert and on query rather than requiring a separate encode step.

Chunking splits on ` ماده ` (*article*) in addition to the usual paragraph and
newline separators. Legal text is article-structured, and splitting on a fixed
token count would cut articles in half — the single most consequential choice in
the pipeline, since an article is the unit a legal question is actually about.

Each chunk carries `filename`, `category`, and `page` metadata for the filtered
retrieval below.

In [ ]:
import lancedb
from lancedb.pydantic import LanceModel, Vector
from lancedb.embeddings import get_registry
from langchain_text_splitters import RecursiveCharacterTextSplitter
import glob
import os
import re 


def clean_legal_text(text):
   
    text = re.sub(r'<\|.*?\|>', '', text)
   
    text = re.sub(r'\[\[.*?\]\]', '', text)
    
    text = re.sub(r'\s+', ' ', text)
    return text.strip()


LOCAL_MODEL_PATH = "intfloat/multilingual-e5-base"
DB_PATH = str(DATA / 'lancedb_store')
DATASET_PATH = str(DATA / 'laws' / '**' / '*.txt')

print(f"Loading embedding model from: {LOCAL_MODEL_PATH}")


db = lancedb.connect(DB_PATH)
registry = get_registry()
func = registry.get("huggingface").create(name=LOCAL_MODEL_PATH)


class LawDoc(LanceModel):
    text: str = func.SourceField()
    vector: Vector(func.ndims()) = func.VectorField()
    filename: str
    category: str 
    chunk_id: int


TABLE_NAME = "iranian_laws"
try:
    db.drop_table(TABLE_NAME)
    print(f"Existing table '{TABLE_NAME}' dropped.")
except:
    pass

table = db.create_table(TABLE_NAME, schema=LawDoc)


text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=200,
    separators=["\n\n", " ماده ", "\n", " "]
)

CATEGORY_MAP = {
    "كار": "Labor_Law",
    "چك": "Check_Law",
    "صدور چک": "Check_Law",
    "ماليات": "Tax_VAT_Law",
    "ارزش افزوده": "Tax_VAT_Law",
    "موجر": "Tenant_Landlord_Law",
    "تأمين اجتماعي": "Social_Security_Law",
    "ثبت": "Registration_Law",
    "اسناد و املاك": "Registration_Law",
    "قاچاق": "Anti_Smuggling_Law",
    "مدنی": "Civil_Law",
    "مجازات": "Criminal_Law"
}


files = glob.glob(DATASET_PATH, recursive=True)
print(f" Found {len(files)} files to process.")

docs_to_add = []

for file_path in files:
   
    with open(file_path, "r", encoding="utf-8") as f:
        content = f.read()
    
    file_name = os.path.basename(file_path)
    detected_category = "General_Law"
    
    for key, tag in CATEGORY_MAP.items():
        if key in file_name:
            detected_category = tag
            break
    
    print(f"Processing: {file_name} -> Tag: {detected_category}")
    
   
    chunks = text_splitter.split_text(content)
    
  
    for i, chunk in enumerate(chunks):
        cleaned_chunk = clean_legal_text(chunk)
        
     
        if len(cleaned_chunk) > 20: 
            docs_to_add.append({
                "text": cleaned_chunk,
                "filename": file_name,
                "category": detected_category,
                "chunk_id": i
            })
                



if docs_to_add:
    print(f" Inserting {len(docs_to_add)} cleaned chunks into LanceDB...")
    table.add(docs_to_add)
    print(" Ingestion complete successfully!")
else:
    print(" No documents were processed.")

In [ ]:
import os

# Credentials are read from the process environment. Export these before
# starting Jupyter; see the README for what each one is and where to get it.
REQUIRED = ["LLM_API_KEY", "OPENAI_API_KEY", "OPENAI_API_BASE"]

missing = [k for k in REQUIRED if not os.environ.get(k)]
if missing:
    raise RuntimeError(
        "Missing environment variables: " + ", ".join(missing)
        + "\nExport them in your shell before launching Jupyter."
    )
print("Credentials found for:", ", ".join(REQUIRED))

## 4. Graph state and credentials

The pipeline's state object and the LLM client. Credentials come from the
environment — see the README for the variable names.

In [ ]:
import os
import lancedb
from typing import TypedDict, List, Literal, Optional
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END
from lancedb.embeddings import get_registry


from langchain_openai import ChatOpenAI
from langchain_ollama import ChatOllama


   
llm = ChatOpenAI(
    model="gpt-4o-mini",
    api_key=os.environ["LLM_API_KEY"],  
    base_url="https://api.avalai.ir/v1",  
    temperature=0.1,
    n=3

)


db_path = str(DATA / 'lancedb_store') 


db = lancedb.connect(db_path)


local_e5_path = "intfloat/multilingual-e5-base" 
registry = get_registry()
func = registry.get("huggingface").create(name=local_e5_path)


class LawDoc(lancedb.pydantic.LanceModel):
    text: str = func.SourceField()
    vector: lancedb.pydantic.Vector(func.ndims()) = func.VectorField()
    filename: str
    category: str
    page: int



table = db.open_table("iranian_laws")
print(f" Successfully connected to database at: {db_path}")
print(f" Found table 'iranian_laws' with {len(table)} rows.")


### The state schema

`AgentState` threads the original query, the rewritten query, extracted filters,
retrieved chunks, and a retry counter through every node.

In [ ]:

class AgentState(TypedDict):
    original_query: str
    rewritten_query: str
    intent: str
    metadata_filters: dict
    retrieved_docs: List[dict] 
    final_docs: List[dict]     
    retry_count: int           
    final_answer: str


class MetadataQuery(BaseModel):
    law_name: Optional[Literal[
        "Labor_Law", "Check_Law", "Tax_VAT_Law", 
        "Tenant_Landlord_Law", "Social_Security_Law", 
        "Registration_Law", "Anti_Smuggling_Law"
    ]] = Field(None, description="The English category tag. MUST be one of the listed values.")
    article_number: Optional[str] = Field(None, description="Just the number.")

class IntentClassifier(BaseModel):
    intent: Literal["Greeting", "Abusive", "Law Question"]

### Persian digit normalization

Article numbers appear as both `۵۱` and `51` depending on the source PDF, so
filters have to be built in both forms. Six lines that decide whether an
article-number filter ever matches.

In [ ]:
def to_persian_num(num_str):
    if not num_str: return None
    english = "0123456789"
    persian = "۰۱۲۳۴۵۶۷۸۹"
    translation_table = str.maketrans(english, persian)
    return num_str.translate(translation_table)

## 5. The six nodes

1. **`rewrite_query`** — restate a colloquial question as formal legal Persian.
2. **`classify_intent`** — Greeting / Abusive / Law Question, so the graph can
   short-circuit non-legal input instead of retrieving against it.
3. **`extract_metadata`** — pull law name and article number into a structured
   filter.
4. **`context_retrieve`** — filtered vector search, falling back to unfiltered.
5. **`rerank`** — relevance check, with a retry edge back to rewriting.
6. **`generate_answer`** — answer strictly from retrieved text, in plain language.

**Two failure modes are visible in the logs and worth naming.**

**The metadata filter never fires.** Every one of the 21 evaluation queries logs
`Article Search failed. Trying Global Search without filters`. `extract_metadata`
*does* find the right values — `{'law_name': 'Labor_Law', 'article_number': '51'}`
on the first query — but by the time retrieval runs, `Filters received: {}`. The
filter is extracted and then lost between nodes, so the LanceDB `where` clause is
never applied and the pipeline degrades to plain vector search on every query.
That is the direct cause of the low answer-relevancy score: retrieval has no way
to prefer the labour-law chunk about article 51 over a similarly-worded chunk from
a different code.

**The rewriter sometimes returns its own commentary.** On several queries the LLM
answers with *"To convert your query into a formal Persian legal search query, you
can phrase it as follows: ..."* instead of just the rewritten query — English
preamble that then goes into the retriever as search text. The prompt asks for a
transformation but does not constrain the output format, and a chat-tuned model
defaults to being conversational. Constrained/structured output would fix it.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
def rewrite_query(state: AgentState):
    print("---  REWRITING & EXTRACTING METADATA ---")
    query = state["original_query"]
    
   
    mapping_instruction = """
       سؤال زیر را به یک عبارت فارسی رسمی و مناسب برای جستجوی حقوقی در پایگاه اسناد تبدیل کن:\n\n{query}

    """
    
   
    structured_llm = llm.with_structured_output(MetadataQuery)
    metadata_result = structured_llm.invoke(f"{mapping_instruction}\n\nUser Query: {query}")
    
    
    filters = {k: v for k, v in metadata_result.model_dump().items() if v is not None}
    print(f" Extracted Filters: {filters}")

    
    prompt = ChatPromptTemplate.from_template("Convert to a formal persian legal search query: {query}")
    rewritten = (prompt | llm | StrOutputParser()).invoke({"query": query})
    print(rewritten)
    return {
        "rewritten_query": rewritten,
        "metadata_filters": filters,
        "retry_count": 0
    }

def classify_intent(state: AgentState):
    print("--- CLASSIFYING INTENT ---")
    query = state["original_query"]
    
    parser = PydanticOutputParser(pydantic_object=IntentClassifier)
    prompt = ChatPromptTemplate.from_template(
            "نوع نیت کاربر را مشخص کن و فقط یکی از این برچسب‌ها را انتخاب کن:\n"
    "- Greeting (سلام یا احوال‌پرسی)\n"
    "- Abusive (توهین‌آمیز یا نامناسب)\n"
    "- Law Question (سؤال حقوقی)\n\n"
    "{format_instructions}\n\n"
    "متن پرسش:\n{query}"
    )
    
    chain = prompt | llm | parser
    try:
        result = chain.invoke({"query": query, "format_instructions": parser.get_format_instructions()})
        intent = result.intent
    except:
        intent = "Law Question" # Fallback
        
    return {"intent": intent}


def extract_metadata(state: AgentState):
    print("--- EXTRACTING METADATA ---")
    query = state["rewritten_query"]
    
    parser = PydanticOutputParser(pydantic_object=MetadataQuery)
    prompt = ChatPromptTemplate.from_template(
        "از پرسش زیر، فراداده ساختاریافته برای فیلتر کردن اسناد حقوقی استخراج کن.\n"
    "اگر به مواردی مانند «حقوق مدنی»، «کیفری»، «ماده قانونی»، «نام قانون» اشاره شده بود، آن‌ها را مشخص کن.\n\n"
    "{format_instructions}\n\n"
    "پرسش:\n{query}"
    )
    
    chain = prompt | llm | parser
    try:
        result = chain.invoke({"query": query, "format_instructions": parser.get_format_instructions()})
        filters = {k: v for k, v in result.dict().items() if v is not None}
    except:
        filters = {}
        
    return {"metadata_filters": filters}


def context_retrieve(state: AgentState):
    print("--- DEBUGGING RETRIEVAL ---")
    query = state["rewritten_query"]
    filters = state["metadata_filters"]
    
   
    print(f"DEBUG: Filters received: {filters}")

    results = []
    if filters.get("article_number"):
        art_num = filters["article_number"]
        p_num = to_persian_num(art_num)
        
      
        where_clause = f"(text LIKE '%ماده {p_num}%' OR text LIKE '%ماده {art_num}%')"
        if filters.get("law_name"):
            where_clause = f"category = '{filters['law_name']}' AND {where_clause}"
        
        print(f"DEBUG: Executing SQL: SELECT * FROM table WHERE {where_clause}")
        
        
        search_results = table.search(query).where(where_clause, prefilter=True).limit(5).to_list()
        print(f"DEBUG: Found {len(search_results)} chunks matching Article {p_num}")
        
        for idx, r in enumerate(search_results):
            print(f"DEBUG: Chunk {idx} snippet: {r['text'][:100]}...")
        
        results = search_results

    
    if not results:
        print(" DEBUG: Article Search failed. Trying Global Search without filters...")
        results = table.search(query).limit(3).to_list()
        print(f"DEBUG: Global Search found {len(results)} chunks.")

    return {"retrieved_docs": [{"text": r["text"], "category": r["category"]} for r in results]}

def rerank(state: AgentState):
    print("--- RERANKING & CHECKING RELEVANCE ---")
    docs = state["retrieved_docs"]
    filters = state["metadata_filters"]
    
    if not docs:
        return {"final_docs": [], "retry_count": state.get("retry_count", 0) + 1}

  
    if filters.get("article_number"):
        print(f" Article {filters['article_number']} requested. Bypassing strict rerank.")
      
        return {"final_docs": docs[:3]} 

   
    return {"final_docs": docs[:3]}


def generate_answer(state: AgentState):
    print("--- GENERATING ANSWER ---")
    query = state["original_query"]
    
  
    docs = state.get("final_docs", []) 
    
    if not docs:
        return {"final_answer": "متاسفانه ماده مورد نظر در اسناد یافت نشد."}

   
    context_str = "\n\n".join([d["text"] for d in docs])
    
    prompt = ChatPromptTemplate.from_template("""
        شما یک دستیار هوشمند هستید که متون حقوقی را به زبانی ساده و شفاف برای کاربران توضیح می‌دهد.
        
        دستورالعمل‌ها:
        ۱. پاسخ شما باید **منحصراً** بر اساس اطلاعات موجود در "متن مرجع" زیر باشد.
        ۲. از به‌کار بردن اصطلاحات پیچیده و سنگین حقوقی خودداری کنید؛ سعی کنید مطلب را به زبان ساده و روان بیان کنید.
        ۳. چیزی از دانسته‌های قبلی خود اضافه نکنید.
        ۴. اگر پاسخ سوال در متن نیست، به صراحت بگویید: «اطلاعاتی در این مورد در متن وجود ندارد.»
        
        متن مرجع:
        {context}
        
        سؤال کاربر:
        {query}
        
        پاسخ ساده و مستند:
        """
    )
    
    chain = prompt | llm | StrOutputParser()
    answer = chain.invoke({"context": context_str, "query": query})
    
    return {"final_answer": answer}

def handle_greeting(state: AgentState):
    return {"final_answer": "سلام! من دستیار حقوقی هوشمند هستم. چطور می‌توانم به شما کمک کنم؟"}

def handle_abusive(state: AgentState):
    return {"final_answer": "من یک مدل هوش مصنوعی هستم که برای پاسخگویی به سوالات حقوقی طراحی شده‌ام و وارد بحث‌های تنش‌زا نمی‌شوم. لطفاً سوال حقوقی خود را بپرسید."}

### Wiring the graph

Nodes, edges, and the conditional retry edge from `rerank` back to
`rewrite_query`.

In [ ]:

workflow = StateGraph(AgentState)

workflow.add_node("rewrite_query", rewrite_query)
workflow.add_node("classify_intent", classify_intent)
workflow.add_node("handle_greeting", handle_greeting)
workflow.add_node("handle_abusive", handle_abusive)
workflow.add_node("extract_metadata", extract_metadata)
workflow.add_node("context_retrieve", context_retrieve)
workflow.add_node("rerank", rerank)            
workflow.add_node("generate_answer", generate_answer)

workflow.set_entry_point("rewrite_query")

workflow.add_edge("rewrite_query", "classify_intent")

def route_intent(state):
    intent = state["intent"]
    if intent == "Greeting": return "handle_greeting"
    elif intent == "Abusive": return "handle_abusive"
    else: return "extract_metadata"

workflow.add_conditional_edges("classify_intent", route_intent, 
    {"handle_greeting": "handle_greeting", "handle_abusive": "handle_abusive", "extract_metadata": "extract_metadata"}
)

workflow.add_edge("extract_metadata", "context_retrieve")
workflow.add_edge("context_retrieve", "rerank")


def check_relevance(state):
    # Logic:
    # retry_count = 0 -> Initial run
    # retry_count = 1 -> First failure (Trigger Retry)
    # retry_count = 2 -> Second failure (Give up and Generate)
    
    if not state["final_docs"] and state["retry_count"] == 1:
        return "retry"
    
    # If success OR if we failed twice (count > 1), go to generate
    return "generate"

workflow.add_conditional_edges(
    "rerank",
    check_relevance,
    {
        "retry": "context_retrieve", 
        "generate": "generate_answer"
    }
)

workflow.add_edge("handle_greeting", END)
workflow.add_edge("handle_abusive", END)
workflow.add_edge("generate_answer", END)

app = workflow.compile()

### A single query end to end

In [ ]:
result = app.invoke({"original_query":"   قانون 10 سر قفلی در اسناد داده شده دقیقا چیست؟", "retry_count": 0})
print(result["final_answer"])

## 6. Where the time goes

Average latency per node across 21 questions:

| Node | Seconds | Share |
|---|---|---|
| `rewrite_query` | 2.510 | 43% |
| `generate_answer` | 1.575 | 27% |
| `extract_metadata` | 1.224 | 21% |
| `classify_intent` | 1.001 | 17% |
| `context_retrieve` | 0.078 | 1.3% |
| `rerank` | 0.000009 | ~0% |

**Retrieval is 1.3% of the wall clock. The four LLM calls are 98%.**

This inverts the intuitive optimisation target. Tuning the vector index would be
almost worthless here; the cost is that the pipeline makes *four sequential LLM
round-trips before generating anything*. `classify_intent` and `extract_metadata`
are independent of each other and could run concurrently, or be folded into one
call with a structured schema, cutting ~2.2s of the ~6.4s total.

`rerank` at 9 microseconds is the other signal: it is not calling a cross-encoder
at all, just checking scores that retrieval already returned. It is named for
something it does not do.

In [ ]:
import time
import pandas as pd
import matplotlib.pyplot as plt
from langgraph.graph import StateGraph, END

test_questions = [
    # 1. Labor Law - based on articles 51, 64, 79
    "ساعات کار موظفی کارگران در هفته طبق ماده ۵۱ چقدر است؟",
    "میزان مرخصی استحقاقی سالانه کارگران با احتساب روزهای جمعه چقدر است؟",
    "حداقل سن قانونی برای اشتغال بکار طبق ماده ۷۹ چه سنی است؟",

    # 2. Cheque Law - based on articles 11, 13
    "مهلت شکایت کیفری برای چک برگشتی از تاریخ گواهی عدم پرداخت چقدر است؟",
    "آیا صدور چک وعده‌دار (مشروط) مشمول مجازات کیفری می‌شود؟",
    "وظیفه بانک در صورت کسر موجودی حساب صادرکننده چک چیست؟",

    # 3. VAT Law - based on exemptions and penalties
    "طبق قانون، پرداخت‌کننده نهایی مالیات بر ارزش افزوده چه کسی است؟",
    "آیا خدمات درمانی و پزشکی مشمول مالیات بر ارزش افزوده هستند؟",
    "جریمه عدم صدور صورتحساب قانونی توسط مودیان چیست؟",

    # 4. Landlord & Tenant Act 1376 - based on articles 2, 3
    "طبق قانون سال ۷۶، قرارداد اجاره باید توسط چند شاهد امضا شود؟",
    "مهلت اجرای دستور تخلیه توسط ضابطین قضایی پس از ابلاغ چقدر است؟",
    "در چه صورتی موجر حق فسخ قرارداد و تخلیه ملک را پیش از موعد دارد؟",

    # 5. Social Security Law - based on articles 28, 76
    "سهم حق بیمه کارفرما طبق ماده ۲۸ قانون تامین اجتماعی چند درصد است؟",
    "شرایط بازنشستگی برای مردان با سابقه پرداخت حق بیمه چیست؟",
    "شرط اصلی دریافت بیمه بیکاری برای کارگر اخراج شده چیست؟",

    # 6. Deeds Registration Law - based on articles 46, 48
    "طبق ماده ۴۸، اسنادی که باید ثبت شوند ولی نشده‌اند در ادارات چه حکمی دارند؟",
    "مهلت اعتراض به ثبت ملک از تاریخ نشر آگهی نوبتی چقدر است؟",
    "تفاوت اعتبار سند رسمی و سند عادی در دادگاه چیست؟",

    # 7. Anti-Smuggling Law - based on penalties
    "مجازات نگهداری کالای قاچاق ممنوع طبق قانون چیست؟",
    "مرجع صالح برای رسیدگی به پرونده‌های قاچاق حرفه‌ای و سازمان‌یافته کجاست؟",
    "حکم قانونی وسیله نقلیه‌ای که برای حمل کالای قاچاق استفاده شده چیست؟"
]


execution_logs = []

def timer_wrapper(node_name, func):
    
    def wrapper(state):
        start_time = time.time()
        try:
            result = func(state)
        except Exception as e:
            print(f"Error in {node_name}: {e}")
            result = {}
        end_time = time.time()
        
        execution_logs.append({
            "question": state.get("original_query", "Unknown"),
            "node": node_name,
            "duration": end_time - start_time
        })
        return result
    return wrapper


workflow_timed = StateGraph(AgentState)


workflow_timed.add_node("rewrite_query", timer_wrapper("rewrite_query", rewrite_query))
workflow_timed.add_node("classify_intent", timer_wrapper("classify_intent", classify_intent))
workflow_timed.add_node("handle_greeting", timer_wrapper("handle_greeting", handle_greeting))
workflow_timed.add_node("handle_abusive", timer_wrapper("handle_abusive", handle_abusive))
workflow_timed.add_node("extract_metadata", timer_wrapper("extract_metadata", extract_metadata))
workflow_timed.add_node("context_retrieve", timer_wrapper("context_retrieve", context_retrieve))
workflow_timed.add_node("rerank", timer_wrapper("rerank", rerank))
workflow_timed.add_node("generate_answer", timer_wrapper("generate_answer", generate_answer))

workflow_timed.set_entry_point("rewrite_query")
workflow_timed.add_edge("rewrite_query", "classify_intent")

# Logic for conditional edges
def route_intent(state):
    intent = state["intent"]
    if intent == "Greeting": return "handle_greeting"
    elif intent == "Abusive": return "handle_abusive"
    else: return "extract_metadata"

workflow_timed.add_conditional_edges("classify_intent", route_intent, 
    {"handle_greeting": "handle_greeting", "handle_abusive": "handle_abusive", "extract_metadata": "extract_metadata"}
)

workflow_timed.add_edge("extract_metadata", "context_retrieve")
workflow_timed.add_edge("context_retrieve", "rerank")

def check_relevance(state):
    if not state.get("final_docs") and state.get("retry_count", 0) == 1:
        return "retry"
    return "generate"

workflow_timed.add_conditional_edges("rerank", check_relevance,
    {"retry": "context_retrieve", "generate": "generate_answer"}
)

workflow_timed.add_edge("handle_greeting", END)
workflow_timed.add_edge("handle_abusive", END)
workflow_timed.add_edge("generate_answer", END)

app_timed = workflow_timed.compile()


print(f"Starting analysis on {len(test_questions)} questions...")
for q in test_questions:
    print(f"Processing: {q}")
    app_timed.invoke({"original_query": q, "retry_count": 0})


df_logs = pd.DataFrame(execution_logs)
mean_times = df_logs.groupby("node")["duration"].mean().sort_values(ascending=False)

print("\n--- Average Time per Node (Seconds) ---")
print(mean_times)

plt.figure(figsize=(10, 6))
mean_times.plot(kind='bar', color='teal', edgecolor='black')
plt.title('Performance Bottleneck Analysis')
plt.ylabel('Time (s)')
plt.xlabel('Node')
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

## 7. RAGAS evaluation

21 questions with hand-written gold answers, three per legal code, scored on:

- **faithfulness** — is the answer supported by the retrieved context?
- **answer relevancy** — does the answer actually address the question?

**Result: faithfulness 0.7341, answer relevancy 0.5140.**

Read together with section 5, the split is coherent and diagnostic. The
generation prompt's grounding instructions are doing their job: 0.73 faithfulness
means the model is largely not inventing legal text, which is the failure mode
that matters most in this domain. But retrieval is running unfiltered on every
query, so the context it grounds *in* is frequently from the wrong code — and
answer relevancy at 0.51 is what that looks like downstream.

The fix is in the plumbing, not the prompts: pass the extracted filters through to
the retrieval node.

In [ ]:
import pandas as pd
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy
from langchain_huggingface import HuggingFaceEmbeddings 
import warnings
warnings.filterwarnings("ignore")

evaluation_data = [
    # 1. Labor Law
    {
        "question": "ساعات کار موظفی کارگران در هفته چقدر است؟",
        "answer": "طبق ماده 51 قانون کار، ساعات کار عادی کارگران نباید از 44 ساعت در هفته تجاوز کند.",
        "source_doc": "نسخه چاپی قانون كار.pdf"
    },
    {
        "question": "میزان مرخصی استحقاقی سالانه کارگران چقدر است؟",
        "answer": "طبق ماده 64، مرخصی استحقاقی سالانه با استفاده از مزد و احتساب چهار روز جمعه، جمعاً یک ماه است.",
        "source_doc": "نسخه چاپی قانون كار.pdf"
    },
    {
        "question": "مرجع تعیین حداقل دستمزد کارگران کیست؟",
        "answer": "طبق ماده 41، شورای عالی کار موظف است همه ساله میزان حداقل مزد کارگران را تعیین نماید.",
        "source_doc": "نسخه چاپی قانون كار.pdf"
    },
    # 2. Check Law
    {
        "question": "مهلت قانونی برای شکایت کیفری چک برگشتی چقدر است؟",
        "answer": "دارنده چک باید ظرف ۶ ماه از تاریخ صدور گواهی عدم پرداخت شکایت نماید.",
        "source_doc": "نسخه چاپی قانون صدور چك_page-0001.pdf"
    },
    {
        "question": "مجازات صدور چک بلامحل چیست؟",
        "answer": "بسته به مبلغ، حبس تعزیری و ممنوعیت از داشتن دسته چک (طبق قانون اصلاحی).",
        "source_doc": "نسخه چاپی قانون صدور چك_page-0001.pdf"
    },
    {
        "question": "آیا چک تضمینی قابل تعقیب کیفری است؟",
        "answer": "خیر، صدور چک به عنوان تضمین یا چک سفید امضا فاقد جنبه کیفری است.",
        "source_doc": "نسخه چاپی قانون صدور چك_page-0001.pdf"
    },
    # 3. VAT Law
    {
        "question": "تعریف «عرضه کالا» در قانون مالیات بر ارزش افزوده چیست؟",
        "answer": "طبق ماده 1، انتقال مالکیت کالا به دیگری از طریق هر نوع معامله، عرضه کالا محسوب می‌شود.",
        "source_doc": "نسخه چاپی قانون ماليات بر ارزش افزوده.pdf"
    },
    {
        "question": "آیا محصولات کشاورزی فرآوری نشده مشمول مالیات هستند؟",
        "answer": "خیر، طبق قانون، عرضه محصولات کشاورزی فرآوری نشده از پرداخت مالیات و عوارض معاف است.",
        "source_doc": "نسخه چاپی قانون ماليات بر ارزش افزوده.pdf"
    },
    {
        "question": "مهلت تسلیم اظهارنامه مالیات بر ارزش افزوده چه زمانی است؟",
        "answer": "معمولاً 15 روز پس از پایان هر فصل (دوره مالیاتی) است.",
        "source_doc": "نسخه چاپی قانون ماليات بر ارزش افزوده.pdf"
    },
    # 4. Landlord & Tenant Law
    {
        "question": "شرایط تخلیه فوری ملک در قانون سال 1376 چیست؟",
        "answer": "قرارداد باید رسمی یا با امضای دو شاهد باشد و مدت اجاره منقضی شده باشد.",
        "source_doc": "نسخه چاپی قانون روابط موجر و مستأجر_page-0001.pdf"
    },
    {
        "question": "مهلت اجرای دستور تخلیه چقدر است؟",
        "answer": "پس از ابلاغ دستور قضایی، ظرف مدت یک هفته اجرا می‌شود.",
        "source_doc": "نسخه چاپی قانون روابط موجر و مستأجر_page-0001.pdf"
    },
    {
        "question": "آیا مستأجر حق دریافت سرقفلی را دارد؟",
        "answer": "تنها در صورتی که در ابتدای اجاره مبلغی به عنوان سرقفلی پرداخت کرده باشد، به نرخ روز دریافت می‌کند.",
        "source_doc": "نسخه چاپی قانون روابط موجر و مستأجر_page-0001.pdf"
    },
    # 5. Social Security Law
    {
        "question": "نرخ حق بیمه سهم کارگر و کارفرما چقدر است؟",
        "answer": "طبق ماده 28، حق بیمه 30٪ مزد است: 7٪ سهم بیمه شده، 20٪ سهم کارفرما و 3٪ کمک دولت.",
        "source_doc": "نسخه چاپی قانون تأمين اجتماعي.pdf"
    },
    {
        "question": "شرایط استفاده از بیمه بیکاری چیست؟",
        "answer": "بیمه‌شده باید بدون میل و اراده بیکار شده و آماده به کار باشد و سابقه پرداخت حق بیمه داشته باشد.",
        "source_doc": "نسخه چاپی قانون تأمين اجتماعي.pdf"
    },
    {
        "question": "سن بازنشستگی برای مردان چقدر است؟",
        "answer": "طبق ماده 76، شصت سال سن برای مردان مشروط بر داشتن حداقل سابقه پرداخت حق بیمه.",
        "source_doc": "نسخه چاپی قانون تأمين اجتماعي.pdf"
    },
    # 6. Registration Law
    {
        "question": "آیا ثبت املاک اجباری است؟",
        "answer": "بله، در نقاطی که اداره ثبت آگهی می‌کند، ثبت کلیه اموال غیرمنقول اجباری است (ماده 46/47).",
        "source_doc": "نسخه چاپی قانون ثبت اسناد و املاك.pdf"
    },
    {
        "question": "اعتبار اسناد ثبت نشده در دادگاه‌ها چگونه است؟",
        "answer": "طبق ماده 48، سندی که باید به ثبت برسد و نرسیده، در هیچ‌یک از ادارات و محاکم پذیرفته نخواهد شد.",
        "source_doc": "نسخه چاپی قانون ثبت اسناد و املاك.pdf"
    },
    {
        "question": "تفاوت سند رسمی با عادی چیست؟",
        "answer": "سند رسمی لازم‌الاجرا است و انکار و تردید نسبت به آن مسموع نیست (فقط ادعای جعل).",
        "source_doc": "نسخه چاپی قانون ثبت اسناد و املاك.pdf"
    },
    # 7. Anti-Smuggling Law
    {
        "question": "تعریف قاچاق کالا و ارز چیست؟",
        "answer": "هر فعل یا ترک فعلی که موجب نقض تشریفات قانونی ورود و خروج کالا و ارز گردد (ماده 1).",
        "source_doc": "نسخه چاپی قانون مبارزه با قاچاق كالا و ارز.pdf"
    },
    {
        "question": "مرجع رسیدگی به جرایم قاچاق سازمان یافته کجاست؟",
        "answer": "در صلاحیت دادسرا و دادگاه انقلاب است.",
        "source_doc": "نسخه چاپی قانون مبارزه با قاچاق كالا و ارز.pdf"
    },
    {
        "question": "مجازات نگهداری کالای قاچاق چیست؟",
        "answer": "علاوه بر ضبط کالا، جریمه نقدی معادل چند برابر ارزش کالا دریافت می‌شود.",
        "source_doc": "نسخه چاپی قانون مبارزه با قاچاق كالا و ارز.pdf"
    }
]

print("Loading embeddings for Ragas evaluation")
ragas_embeddings = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-base"  
)


ragas_data = {
    "question": [],     
    "ground_truth": [],  
    "answer": [],        
    "contexts": []       
}

print(f"Starting Evaluation on {len(evaluation_data)} samples...")

for i, item in enumerate(evaluation_data):
   
    q = item["question"]
    true_answer = item["answer"] 
    
    print(f"Processing ({i+1}/{len(evaluation_data)}): {q}")
    
    ragas_data["question"].append(q)
    ragas_data["ground_truth"].append(true_answer) 
    
    try:
       
        result = app.invoke({"original_query": q, "retry_count": 0})
        
 
        agent_answer = result.get("final_answer", "No answer generated.")
        
     
        final_docs = result.get("final_docs", [])
        if final_docs:
            if isinstance(final_docs[0], dict):
                
                retrieved_texts = [d.get("page_content", d.get("text", "")) for d in final_docs]
            else:
                 
                retrieved_texts = [d.page_content for d in final_docs]
        else:
            retrieved_texts = ["No context retrieved."]
            
    except Exception as e:
        print(f"Error processing question: {e}")
        agent_answer = "Error occurred."
        retrieved_texts = ["Error in retrieval."]

    ragas_data["answer"].append(agent_answer)
    ragas_data["contexts"].append(retrieved_texts)


print("\n Calculating Ragas Metrics...")
dataset = Dataset.from_dict(ragas_data)

results = evaluate(
    dataset=dataset,
    metrics=[faithfulness, answer_relevancy],
    llm=llm,                
    embeddings=ragas_embeddings
)

print("\nEvaluation Results:")


print(results)


### Exporting the results

In [ ]:
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Alignment

df = pd.DataFrame({
    "Question": ragas_data["question"],
    "Retrieved Context": ["\n---\n".join(ctx) for ctx in ragas_data["contexts"]],
    "Agent Answer": ragas_data["answer"],
    "Ground Truth": ragas_data["ground_truth"],
    "Faithfulness": results['faithfulness'],
    "Answer Relevancy": results['answer_relevancy']
})


df.to_excel("evaluation_results.xlsx", index=False)



## 8. A Chainlit front end

The same graph packaged as a chat application, written out to `app.py`. Run it
with `chainlit run app.py -w`.

In [ ]:
%%writefile app.py
import os
import chainlit as cl
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END
import lancedb
from lancedb.embeddings import get_registry
from typing import TypedDict, List, Optional, Literal
from langchain_core.messages import HumanMessage
from pydantic import BaseModel, Field
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser
from langchain_core.prompts import ChatPromptTemplate


LOCAL_MODEL_PATH = "intfloat/multilingual-e5-base"
DB_PATH = str(DATA / 'lancedb_store')


os.environ["OPENAI_API_KEY"] = os.environ["HF_TOKEN"] 
os.environ["OPENAI_API_BASE"] = "https://api.avalai.ir/v1"


db = lancedb.connect(DB_PATH)
registry = get_registry()
func = registry.get("huggingface").create(name=LOCAL_MODEL_PATH)
table = db.open_table("iranian_laws")


llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.0)


def to_persian_num(num_str):
    if not num_str: return None
    english = "0123456789"
    persian = "۰۱۲۳۴۵۶۷۸۹"
    translation_table = str.maketrans(english, persian)
    return num_str.translate(translation_table)


class AgentState(TypedDict):
    original_query: str
    rewritten_query: str
    intent: str
    metadata_filters: dict
    retrieved_docs: List[dict]
    final_docs: List[dict]
    retry_count: int
    final_answer: str

class MetadataQuery(BaseModel):
    law_name: Optional[Literal[
        "Labor_Law", "Check_Law", "Tax_VAT_Law", 
        "Tenant_Landlord_Law", "Social_Security_Law", 
        "Registration_Law", "Anti_Smuggling_Law"
    ]] = Field(None, description="The English category tag.")
    article_number: Optional[str] = Field(None, description="Just the number.")

class IntentClassifier(BaseModel):
    intent: Literal["Greeting", "Abusive", "Law Question"]



def rewrite_query(state: AgentState):
    print("---  REWRITING & EXTRACTING METADATA ---")
    query = state["original_query"]
    
   
    mapping_instruction = """
       سؤال زیر را به یک عبارت فارسی رسمی و مناسب برای جستجوی حقوقی در پایگاه اسناد تبدیل کن:\n\n{query}

    """
    
   
    structured_llm = llm.with_structured_output(MetadataQuery)
    metadata_result = structured_llm.invoke(f"{mapping_instruction}\n\nUser Query: {query}")
    
    
    filters = {k: v for k, v in metadata_result.model_dump().items() if v is not None}
    print(f" Extracted Filters: {filters}")

    
    prompt = ChatPromptTemplate.from_template("Convert to a formal persian legal search query: {query}")
    rewritten = (prompt | llm | StrOutputParser()).invoke({"query": query})
    print(rewritten)
    return {
        "rewritten_query": rewritten,
        "metadata_filters": filters,
        "retry_count": 0
    }

def classify_intent(state: AgentState):
    print("--- CLASSIFYING INTENT ---")
    query = state["original_query"]
    
    parser = PydanticOutputParser(pydantic_object=IntentClassifier)
    prompt = ChatPromptTemplate.from_template(
            "نوع نیت کاربر را مشخص کن و فقط یکی از این برچسب‌ها را انتخاب کن:\n"
    "- Greeting (سلام یا احوال‌پرسی)\n"
    "- Abusive (توهین‌آمیز یا نامناسب)\n"
    "- Law Question (سؤال حقوقی)\n\n"
    "{format_instructions}\n\n"
    "متن پرسش:\n{query}"
    )
    
    chain = prompt | llm | parser
    try:
        result = chain.invoke({"query": query, "format_instructions": parser.get_format_instructions()})
        intent = result.intent
    except:
        intent = "Law Question" 
        
    return {"intent": intent}


def extract_metadata(state: AgentState):
    print("--- EXTRACTING METADATA ---")
    query = state["rewritten_query"]
    
    parser = PydanticOutputParser(pydantic_object=MetadataQuery)
    prompt = ChatPromptTemplate.from_template(
        "از پرسش زیر، فراداده ساختاریافته برای فیلتر کردن اسناد حقوقی استخراج کن.\n"
    "اگر به مواردی مانند «حقوق مدنی»، «کیفری»، «ماده قانونی»، «نام قانون» اشاره شده بود، آن‌ها را مشخص کن.\n\n"
    "{format_instructions}\n\n"
    "پرسش:\n{query}"
    )
    
    chain = prompt | llm | parser
    try:
        result = chain.invoke({"query": query, "format_instructions": parser.get_format_instructions()})
        filters = {k: v for k, v in result.dict().items() if v is not None}
    except:
        filters = {}
        
    return {"metadata_filters": filters}


def context_retrieve(state: AgentState):
    print("--- DEBUGGING RETRIEVAL ---")
    query = state["rewritten_query"]
    filters = state["metadata_filters"]
    
   
    print(f"DEBUG: Filters received: {filters}")

    results = []
    if filters.get("article_number"):
        art_num = filters["article_number"]
        p_num = to_persian_num(art_num)
        
      
        where_clause = f"(text LIKE '%ماده {p_num}%' OR text LIKE '%ماده {art_num}%')"
        if filters.get("law_name"):
            where_clause = f"category = '{filters['law_name']}' AND {where_clause}"
        
        print(f"DEBUG: Executing SQL: SELECT * FROM table WHERE {where_clause}")
        
        
        search_results = table.search(query).where(where_clause, prefilter=True).limit(5).to_list()
        print(f"DEBUG: Found {len(search_results)} chunks matching Article {p_num}")
        
        for idx, r in enumerate(search_results):
            print(f"DEBUG: Chunk {idx} snippet: {r['text'][:100]}...")
        
        results = search_results

    
    if not results:
        print(" DEBUG: Article Search failed. Trying Global Search without filters...")
        results = table.search(query).limit(3).to_list()
        print(f"DEBUG: Global Search found {len(results)} chunks.")

    return {"retrieved_docs": [{"text": r["text"], "category": r["category"]} for r in results]}

def rerank(state: AgentState):
    print("--- RERANKING & CHECKING RELEVANCE ---")
    docs = state["retrieved_docs"]
    filters = state["metadata_filters"]
    
    if not docs:
        return {"final_docs": [], "retry_count": state.get("retry_count", 0) + 1}

  
    if filters.get("article_number"):
        print(f" Article {filters['article_number']} requested. Bypassing strict rerank.")
      
        return {"final_docs": docs[:3]} 

   
    return {"final_docs": docs[:3]}


def generate_answer(state: AgentState):
    print("--- GENERATING ANSWER ---")
    query = state["original_query"]
    
  
    docs = state.get("final_docs", []) 
    
    if not docs:
        return {"final_answer": "متاسفانه ماده مورد نظر در اسناد یافت نشد."}

   
    context_str = "\n\n".join([d["text"] for d in docs])
    
    prompt = ChatPromptTemplate.from_template("""
        شما یک دستیار هوشمند هستید که متون حقوقی را به زبانی ساده و شفاف برای کاربران توضیح می‌دهد.
        
        دستورالعمل‌ها:
        ۱. پاسخ شما باید **منحصراً** بر اساس اطلاعات موجود در "متن مرجع" زیر باشد.
        ۲. از به‌کار بردن اصطلاحات پیچیده و سنگین حقوقی خودداری کنید؛ سعی کنید مطلب را به زبان ساده و روان بیان کنید.
        ۳. چیزی از دانسته‌های قبلی خود اضافه نکنید.
        ۴. اگر پاسخ سوال در متن نیست، به صراحت بگویید: «اطلاعاتی در این مورد در متن وجود ندارد.»
        
        متن مرجع:
        {context}
        
        سؤال کاربر:
        {query}
        
        پاسخ ساده و مستند:
        """
    )
    
    chain = prompt | llm | StrOutputParser()
    answer = chain.invoke({"context": context_str, "query": query})
    
    return {"final_answer": answer}

def handle_greeting(state: AgentState):
    return {"final_answer": "سلام! من دستیار حقوقی هوشمند هستم. چطور می‌توانم به شما کمک کنم؟"}

def handle_abusive(state: AgentState):
    return {"final_answer": "من یک مدل هوش مصنوعی هستم که برای پاسخگویی به سوالات حقوقی طراحی شده‌ام و وارد بحث‌های تنش‌زا نمی‌شوم. لطفاً سوال حقوقی خود را بپرسید."}


workflow = StateGraph(AgentState)

workflow.add_node("rewrite_query", rewrite_query)
workflow.add_node("classify_intent", classify_intent)
workflow.add_node("handle_greeting", handle_greeting)
workflow.add_node("handle_abusive", handle_abusive)
workflow.add_node("extract_metadata", extract_metadata)
workflow.add_node("context_retrieve", context_retrieve)
workflow.add_node("rerank", rerank)
workflow.add_node("generate_answer", generate_answer)

workflow.set_entry_point("rewrite_query")
workflow.add_edge("rewrite_query", "classify_intent")

def route_intent(state):
    if state["intent"] == "Greeting": return "handle_greeting"
    elif state["intent"] == "Abusive": return "handle_abusive"
    else: return "extract_metadata"

workflow.add_conditional_edges("classify_intent", route_intent, 
    {"handle_greeting": "handle_greeting", "handle_abusive": "handle_abusive", "extract_metadata": "extract_metadata"})

workflow.add_edge("extract_metadata", "context_retrieve")
workflow.add_edge("context_retrieve", "rerank")

def check_relevance(state):
    if not state.get("final_docs") and state.get("retry_count", 0) < 1:
        return "retry"
    return "generate"

workflow.add_conditional_edges("rerank", check_relevance,
    {"retry": "context_retrieve", "generate": "generate_answer"})

workflow.add_edge("handle_greeting", END)
workflow.add_edge("handle_abusive", END)
workflow.add_edge("generate_answer", END)

app = workflow.compile()


@cl.on_chat_start
async def start():
    cl.user_session.set("graph", app)
    await cl.Message("سلام! سوال حقوقی خود را بپرسید.").send()

@cl.on_message
async def main(message: cl.Message):
    app_instance = cl.user_session.get("graph")
    inputs = {"original_query": message.content, "retry_count": 0}
    result = app_instance.invoke(inputs)
    await cl.Message(content=result["final_answer"]).send()
    
    if result.get("final_docs"):
        sources = "\n".join([f"- {d['category']}: {d['text'][:80]}..." for d in result["final_docs"]])
        await cl.Message(content=f" منابع:\n{sources}").send()